# Notebook 2: LLM Translation and Evaluation

This notebook demonstrates **pure LLM translation** and **decoupled evaluation**.

**Key concepts:**
1. **Translation** - LLM converts narrative to config (single prompt, no tools)
2. **Verification** - Run actual portfolio optimization to prove translation works
3. **Evaluation** - Completely separate step: field accuracy + LLM-as-judge

**All functions are traced to Langfuse via `@observe()` decorator.**

## Setup

In [ ]:
import os
import json
import warnings
from typing import Dict, Any, List
from dotenv import load_dotenv

load_dotenv()
load_dotenv(dotenv_path='../.env')

# Import TRANSLATION functions
from llm_utils import (
    translate_narrative,
    PortfolioConfigSchema,
    LLMProvider
)

# Import EVALUATION functions (completely separate from translation)
from llm_utils import (
    compute_field_accuracy,
    llm_as_judge,
    evaluate_single,
    flush_langfuse
)

# Import LANGFUSE NATIVE dataset and scoring functions
from llm_utils import (
    create_langfuse_dataset,
    run_dataset_experiment
)

# Import portfolio optimization for verification
from portfolio_optimizer import (
    PortfolioConfig, download_market_data, optimize_portfolio,
    optimize_hrp, backtest_portfolio, get_universe
)

warnings.filterwarnings('ignore')

with open('scenarios.json', 'r') as f:
    SCENARIOS = json.load(f)

with open('evaluation_dataset.json', 'r') as f:
    EVAL_DATA = json.load(f)

DEFAULT_PROVIDER = LLMProvider.GEMINI

print("Setup complete!")
print(f"Default LLM Provider: {DEFAULT_PROVIDER.value}")
print(f"Loaded {len(SCENARIOS['personas'])} personas")
print(f"Loaded {len(EVAL_DATA['items'])} evaluation items")
print("\nAll functions traced to Langfuse via @observe()")

## Part 1: Translation

The `translate_narrative()` function:
- Takes an investor narrative as input
- Uses a single LLM call with structured output
- Returns a portfolio configuration JSON
- **No tools, no agents** - pure prompt-based translation

In [2]:
# Show the configuration schema
print("Portfolio Configuration Schema:")
schema = PortfolioConfigSchema.model_json_schema()
print(f"Required fields: {schema.get('required', [])}")
print(f"\nField descriptions:")
for field, props in schema.get('properties', {}).items():
    desc = props.get('description', 'N/A')[:60]
    print(f"  {field}: {desc}")

Portfolio Configuration Schema:
Required fields: ['optimization_target', 'universe', 'time_horizon_years', 'risk_tolerance', 'reasoning']

Field descriptions:
  optimization_target: Optimization objective: min_volatility for low risk, max_sha
  universe: Asset universe to invest in
  time_horizon_years: Investment horizon in years
  max_position: Maximum weight per position (0.05-1.0), null if not specifie
  risk_tolerance: Risk tolerance level inferred from narrative
  allow_short: Whether short selling is allowed
  target_return: Target return for efficient_return optimization, null otherw
  reasoning: Brief explanation of why these parameters were chosen


In [3]:
# Translate Marcus's narrative (aggressive growth)
marcus = SCENARIOS['personas']['marcus_growth']
print(f"PERSONA: {marcus['name']}")
print(f"NARRATIVE: {marcus['narrative']}")
print("\n" + "="*60)
print("TRANSLATING...")

marcus_translation = translate_narrative(
    narrative=marcus['narrative'],
    provider=DEFAULT_PROVIDER,
    session_id="nb2_translation"
)

if marcus_translation['status'] == 'success':
    print("\nTRANSLATION RESULT:")
    print(json.dumps(marcus_translation['config'], indent=2))
else:
    print(f"Error: {marcus_translation['error']}")

PERSONA: Marcus Johnson
NARRATIVE: I'm Marcus, 28 years old, just started my career as a software engineer. I have $50,000 to invest and want to maximize long-term growth. I can tolerate high volatility since I won't need this money for 30+ years. I believe in technology and want concentrated exposure to tech stocks.

TRANSLATING...


I0000 00:00:1768478866.889370 29537767 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers



TRANSLATION RESULT:
{
  "optimization_target": "max_return",
  "universe": "us_tech",
  "time_horizon_years": 30,
  "max_position": null,
  "risk_tolerance": "high",
  "allow_short": false,
  "target_return": null,
  "reasoning": "Marcus is young, has a long time horizon, and wants to maximize growth with high risk tolerance and concentrated tech exposure."
}


In [4]:
# Translate Sarah's narrative (conservative)
sarah = SCENARIOS['personas']['sarah_conservative']
print(f"PERSONA: {sarah['name']}")
print(f"NARRATIVE: {sarah['narrative']}")
print("\n" + "="*60)
print("TRANSLATING...")

sarah_translation = translate_narrative(
    narrative=sarah['narrative'],
    provider=DEFAULT_PROVIDER,
    session_id="nb2_translation"
)

if sarah_translation['status'] == 'success':
    print("\nTRANSLATION RESULT:")
    print(json.dumps(sarah_translation['config'], indent=2))
else:
    print(f"Error: {sarah_translation['error']}")

PERSONA: Sarah Chen
NARRATIVE: I'm Sarah, 58 years old, and planning to retire in 7 years. I have $500,000 saved and want stable income with capital preservation. I can't afford to lose more than 15% of my portfolio. I prefer US-based bond investments and don't want emerging market exposure.

TRANSLATING...

TRANSLATION RESULT:
{
  "optimization_target": "min_volatility",
  "universe": "conservative",
  "time_horizon_years": 7,
  "max_position": null,
  "risk_tolerance": "low",
  "allow_short": false,
  "target_return": null,
  "reasoning": "Sarah is nearing retirement with a short time horizon and a strong need for capital preservation and income. A conservative, low-volatility approach focused on US-based bonds is appropriate given her risk tolerance and aversion to emerging markets."
}


In [5]:
# Translate Elena's narrative (balanced)
elena = SCENARIOS['personas']['elena_balanced']
print(f"PERSONA: {elena['name']}")
print(f"NARRATIVE: {elena['narrative']}")
print("\n" + "="*60)
print("TRANSLATING...")

elena_translation = translate_narrative(
    narrative=elena['narrative'],
    provider=DEFAULT_PROVIDER,
    session_id="nb2_translation"
)

if elena_translation['status'] == 'success':
    print("\nTRANSLATION RESULT:")
    print(json.dumps(elena_translation['config'], indent=2))
else:
    print(f"Error: {elena_translation['error']}")

PERSONA: Elena Rodriguez
NARRATIVE: I'm Elena, 38 years old, and I want to build a portfolio for my children's education fund. My kids are 2 and 5 years old, so I have about 18 years. I have $150,000 to invest with moderate risk tolerance. I'd like global diversification and reasonable risk-adjusted returns. No single position should exceed 15% of the portfolio.

TRANSLATING...

TRANSLATION RESULT:
{
  "optimization_target": "max_sharpe",
  "universe": "global_diversified",
  "time_horizon_years": 18,
  "max_position": 0.15,
  "risk_tolerance": "medium",
  "allow_short": false,
  "target_return": null,
  "reasoning": "Elena has a long time horizon (18 years) and a moderate risk tolerance. A globally diversified portfolio optimized for maximum Sharpe ratio provides a balance between risk and return, suitable for long-term growth. The 15% max position limit ensures diversification."
}


## Part 2: Verification via Portfolio Optimization

To prove the translation produces usable configs, we run actual portfolio optimization.

In [6]:
def run_optimization(config: Dict, persona_name: str = "Unknown") -> Dict[str, Any]:
    """Run portfolio optimization using a translated config."""
    print(f"\n{'='*60}")
    print(f"OPTIMIZING PORTFOLIO FOR: {persona_name}")
    print(f"{'='*60}")
    
    print(f"\nConfig:")
    print(f"  Universe: {config['universe']}")
    print(f"  Target: {config['optimization_target']}")
    print(f"  Risk tolerance: {config['risk_tolerance']}")
    
    # Get market data
    universe = config['universe']
    tickers = get_universe(universe)
    print(f"\nDownloading data for {len(tickers)} assets...")
    
    START_DATE = "2019-01-01"
    END_DATE = "2024-01-01"
    prices = download_market_data(tickers, START_DATE, END_DATE)
    
    # Create portfolio config
    portfolio_config = PortfolioConfig(
        tickers=list(prices.columns),
        start_date=START_DATE,
        end_date=END_DATE,
        optimization_target=config['optimization_target'],
        max_position=config.get('max_position'),
        target_return=config.get('target_return'),
        weight_bounds=(-1, 1) if config.get('allow_short') else (0, 1)
    )
    
    # Optimize
    print("Running optimization...")
    try:
        portfolio = optimize_portfolio(portfolio_config, prices)
    except Exception as e:
        print(f"MVO failed ({e}), using HRP...")
        portfolio = optimize_hrp(prices)
    
    # Backtest
    backtest = backtest_portfolio(portfolio['weights'], prices)
    
    # Results
    print(f"\n{'-'*40}")
    print("RESULTS:")
    print(f"{'-'*40}")
    print(f"  Expected Return: {portfolio['expected_return']*100:.2f}%")
    print(f"  Volatility: {portfolio['volatility']*100:.2f}%")
    print(f"  Sharpe Ratio: {portfolio['sharpe_ratio']:.3f}")
    print(f"  Backtest Sharpe: {backtest['sharpe_ratio']:.3f}")
    print(f"  Max Drawdown: {backtest['max_drawdown']*100:.2f}%")
    
    print(f"\n  Top Holdings:")
    sorted_weights = sorted(portfolio['weights'].items(), key=lambda x: -x[1])
    for ticker, weight in sorted_weights[:5]:
        if weight > 0.01:
            print(f"    {ticker}: {weight*100:.1f}%")
    
    return {"portfolio": portfolio, "backtest": backtest}

print("run_optimization function defined!")

run_optimization function defined!


In [7]:
# Verify Marcus's translation works
if marcus_translation['status'] == 'success':
    marcus_portfolio = run_optimization(marcus_translation['config'], marcus['name'])


OPTIMIZING PORTFOLIO FOR: Marcus Johnson

Config:
  Universe: us_tech
  Target: max_return
  Risk tolerance: high

Running optimization...
MVO failed (target_volatility required for max_return optimization), using HRP...

----------------------------------------
RESULTS:
----------------------------------------
  Expected Return: 25.84%
  Volatility: 25.14%
  Sharpe Ratio: 1.028
  Backtest Sharpe: 1.028
  Max Drawdown: -33.78%

  Top Holdings:
    IBM: 17.7%
    ORCL: 12.9%
    CSCO: 10.6%
    MSFT: 6.5%
    GOOG: 6.3%


In [8]:
# Verify Sarah's translation works
if sarah_translation['status'] == 'success':
    sarah_portfolio = run_optimization(sarah_translation['config'], sarah['name'])


OPTIMIZING PORTFOLIO FOR: Sarah Chen

Config:
  Universe: conservative
  Target: min_volatility
  Risk tolerance: low

Running optimization...

----------------------------------------
RESULTS:
----------------------------------------
  Expected Return: 0.27%
  Volatility: 5.67%
  Sharpe Ratio: -0.305
  Backtest Sharpe: 0.081
  Max Drawdown: -18.11%

  Top Holdings:
    GOVT: 49.4%
    MBB: 31.6%
    VMBS: 19.0%


In [9]:
# Verify Elena's translation works
if elena_translation['status'] == 'success':
    elena_portfolio = run_optimization(elena_translation['config'], elena['name'])


OPTIMIZING PORTFOLIO FOR: Elena Rodriguez

Config:
  Universe: global_diversified
  Target: max_sharpe
  Risk tolerance: medium

Running optimization...

----------------------------------------
RESULTS:
----------------------------------------
  Expected Return: 7.84%
  Volatility: 9.25%
  Sharpe Ratio: 0.631
  Backtest Sharpe: 0.932
  Max Drawdown: -15.40%

  Top Holdings:
    AGG: 15.0%
    DBC: 15.0%
    GLD: 15.0%
    GOVT: 15.0%
    SPY: 15.0%


## Part 3: Evaluation (Decoupled)

Evaluation is **completely separate** from translation:
- `compute_field_accuracy()` - compares predicted vs expected configs
- `llm_as_judge()` - uses LLM to score translation quality
- `evaluate_single()` - runs both evaluations together

**Each function has its own Langfuse trace.**

In [10]:
# Get a test item from evaluation dataset
test_item = EVAL_DATA['items'][0]
narrative = test_item['input']['narrative']
expected = test_item['expected_output']

print("TEST ITEM:")
print(f"Narrative: {narrative[:100]}...")
print(f"\nExpected config:")
print(json.dumps(expected, indent=2))

TEST ITEM:
Narrative: I'm Marcus, 28 years old, just started my career as a software engineer. I have $50,000 to invest an...

Expected config:
{
  "optimization_target": "max_sharpe",
  "universe": "us_tech",
  "time_horizon_years": 30,
  "risk_tolerance": "high",
  "allow_short": false
}


In [11]:
# STEP 1: Translate (separate from evaluation)
print("STEP 1: TRANSLATION")
print("="*60)

translation_result = translate_narrative(
    narrative=narrative,
    provider=DEFAULT_PROVIDER,
    session_id="nb2_eval_demo"
)

if translation_result['status'] == 'success':
    predicted = translation_result['config']
    print("Predicted config:")
    print(json.dumps(predicted, indent=2))
else:
    print(f"Translation failed: {translation_result['error']}")
    predicted = None

STEP 1: TRANSLATION
Predicted config:
{
  "optimization_target": "max_return",
  "universe": "us_tech",
  "time_horizon_years": 30,
  "max_position": null,
  "risk_tolerance": "high",
  "allow_short": false,
  "target_return": null,
  "reasoning": "Marcus is young, has a long time horizon, and wants to maximize growth with high risk tolerance and concentrated tech exposure."
}


In [12]:
# STEP 2a: Evaluate with field accuracy (decoupled)
if predicted:
    print("STEP 2a: FIELD ACCURACY EVALUATION")
    print("="*60)
    
    field_accuracy = compute_field_accuracy(
        predicted=predicted,
        expected=expected,
        session_id="nb2_eval_demo"
    )
    
    print(f"Overall accuracy: {field_accuracy['overall_accuracy']:.2%}")
    print("\nField-by-field:")
    for field, score in field_accuracy.items():
        if field != 'overall_accuracy':
            status = "✓" if score == 1.0 else "✗"
            print(f"  {status} {field}: {score}")

STEP 2a: FIELD ACCURACY EVALUATION
Overall accuracy: 80.00%

Field-by-field:
  ✗ optimization_target_match: 0.0
  ✓ universe_match: 1.0
  ✓ risk_tolerance_match: 1.0
  ✓ allow_short_match: 1.0
  ✓ time_horizon_years_match: 1.0


In [13]:
# STEP 2b: Evaluate with LLM-as-judge (decoupled)
if predicted:
    print("STEP 2b: LLM-AS-JUDGE EVALUATION")
    print("="*60)
    
    llm_scores = llm_as_judge(
        narrative=narrative,
        predicted=predicted,
        expected=expected,
        provider=DEFAULT_PROVIDER,
        session_id="nb2_eval_demo"
    )
    
    print("LLM Judge Scores:")
    for key, value in llm_scores.items():
        print(f"  {key}: {value}")

STEP 2b: LLM-AS-JUDGE EVALUATION
LLM Judge Scores:
  optimization_target_score: 6
  universe_score: 10
  risk_assessment_score: 10
  constraints_score: 8
  overall_score: 8.5
  feedback: Optimization target should be max_sharpe, not max_return, to balance risk and return. Other parameters are well-translated.


In [14]:
# Combined evaluation using evaluate_single()
if predicted:
    print("COMBINED EVALUATION (evaluate_single)")
    print("="*60)
    
    full_eval = evaluate_single(
        narrative=narrative,
        predicted=predicted,
        expected=expected,
        provider=DEFAULT_PROVIDER,
        session_id="nb2_eval_demo"
    )
    
    print(f"\nSummary:")
    print(f"  Field Accuracy: {full_eval['overall_field_accuracy']:.2%}")
    print(f"  LLM Score: {full_eval['overall_llm_score']:.1f}/10")

COMBINED EVALUATION (evaluate_single)

Summary:
  Field Accuracy: 80.00%
  LLM Score: 8.5/10


## Part 4: Langfuse Native Dataset Evaluation

Using **Langfuse native features** for batch evaluation:
- `create_langfuse_dataset()` - Upload evaluation items to Langfuse dataset
- `run_dataset_experiment()` - Run evaluation with automatic trace linking and native scoring

**Key benefits of native Langfuse datasets:**
1. **Dataset versioning** - Track evaluation data in Langfuse
2. **Experiment runs** - Compare different models/prompts on same dataset
3. **Native scoring** - Scores attached directly to traces via `score_trace()`
4. **Dashboard analytics** - View results aggregated by run in Langfuse UI

In [ ]:
# Step 1: Create a Langfuse dataset from evaluation data
print("CREATING LANGFUSE DATASET")
print("="*60)

# Prepare items in Langfuse format
dataset_items = [
    {
        "input": {"narrative": item['input']['narrative']},
        "expected_output": item['expected_output']
    }
    for item in EVAL_DATA['items'][:5]  # Use first 5 items
]

DATASET_NAME = "portfolio-translation-eval"

try:
    dataset = create_langfuse_dataset(
        dataset_name=DATASET_NAME,
        items=dataset_items,
        description="Portfolio translation evaluation - investor narratives to configs"
    )
except Exception as e:
    print(f"Note: {e}")
    print("Dataset may already exist - proceeding with experiment run")

In [ ]:
# Step 2: Run experiment with native Langfuse scoring
print("RUNNING DATASET EXPERIMENT")
print("="*60)

# Each item.run() creates a trace linked to the dataset item
# root_span.score_trace() adds native Langfuse scores

experiment_results = run_dataset_experiment(
    dataset_name=DATASET_NAME,
    run_name="gemini-2.0-flash-v1",
    provider=DEFAULT_PROVIDER,
    run_description="Baseline evaluation with Gemini 2.0 Flash"
)

In [ ]:
# View detailed results
print("DETAILED RESULTS")
print("="*60)

for result in experiment_results['detailed_results']:
    status = "✓" if result['status'] == 'success' else "✗"
    if result['status'] == 'success':
        print(f"{status} Item {result['index']+1}: "
              f"Field accuracy: {result['field_accuracy']:.2%}, "
              f"LLM score: {result['llm_score']:.1f}/10")
    else:
        print(f"{status} Item {result['index']+1}: FAILED - {result.get('error', 'Unknown error')}")

print(f"\n{'='*60}")
print("NATIVE LANGFUSE SCORES RECORDED:")
print("="*60)
print("  - field_accuracy (0-1): Programmatic field comparison")
print("  - llm_judge_score (0-1): Overall LLM judge assessment")
print("  - optimization_target_score (0-1): Target selection quality")
print("  - universe_score (0-1): Asset universe appropriateness")
print("  - risk_assessment_score (0-1): Risk tolerance accuracy")
print("  - constraints_score (0-1): Constraint handling quality")
print(f"\nView in Langfuse: Datasets > {DATASET_NAME} > Runs > {experiment_results['run_name']}")

## Key Takeaways

### Translation (Part 1)
- `translate_narrative()` - single LLM call with JSON parsing + Pydantic validation
- No tools or agents - pure prompt-based translation
- Returns valid JSON matching schema

### Verification (Part 2)
- Run actual portfolio optimization with translated config
- Proves translation produces usable parameters

### Evaluation (Part 3) - **Completely Decoupled**
- `compute_field_accuracy()` - programmatic comparison
- `llm_as_judge()` - LLM-based qualitative scoring
- `evaluate_single()` - runs both together
- All functions take pre-computed predictions as input

### Langfuse Native Dataset Evaluation (Part 4)
- `create_langfuse_dataset()` - Upload evaluation items to Langfuse
- `run_dataset_experiment()` - Run with native features:
  - `item.run()` context manager for automatic trace linking
  - `root_span.score_trace()` for native scoring
- **Dashboard integration**: View experiments, compare runs, analyze scores
- **Scores recorded**: field_accuracy, llm_judge_score, dimension scores

In [ ]:
# Flush Langfuse events
flush_langfuse()
print("Langfuse events flushed!")
print("\nCheck Langfuse dashboard for:")
print("\n1. TRACES (Observability):")
print("   - translate_narrative: Translation traces")
print("   - compute_field_accuracy: Field comparison traces")
print("   - llm_as_judge: LLM evaluation traces")
print("   - evaluate_single: Combined evaluation traces")
print("\n2. DATASETS (Evaluation):")
print(f"   - Dataset: {DATASET_NAME}")
print(f"   - Run: {experiment_results['run_name']}")
print("   - Native scores attached to each trace")
print("\n3. SCORES (per trace):")
print("   - field_accuracy, llm_judge_score")
print("   - optimization_target_score, universe_score")
print("   - risk_assessment_score, constraints_score")